In [ ]:
import cv2
import numpy as np
import os
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Optional, List, Dict
from PIL import Image, ImageDraw, ImageFont

# ==================== PHASE 1: YOLO CLASS DEFINITIONS & DIRS ====================

YOLO_CLASSES = {
    0:  "Player",              # Player's own champion
    1:  "Player+Yuumi",        # Player's champion with Yuumi attached
    2:  "Ally",                # Ally champion
    3:  "Ally+Yuumi",          # Ally champion with Yuumi attached
    4:  "Enemy",               # Enemy champion
    5:  "Enemy+Yuumi",         # Enemy champion with Yuumi attached
    6:  "Ally Turret",         # Ally turret
    7:  "Enemy Turret",        # Enemy turret
    8:  "Ally Structure",      # Ally inhibitor / nexus
    9:  "Enemy Structure",     # Enemy inhibitor / nexus
    10: "Jungle Monster",      # Regular jungle camp (big)
    11: "Epic Monster",        # Baron, Dragon, Herald, etc.
    12: "Ally Minion",         # Allied lane minion
    13: "Enemy Minion",        # Enemy lane minion
    14: "Mini Jungle Monster", # Small jungle camp monsters (mini1 / mini2)
}

CLASS_TO_ID = {v: k for k, v in YOLO_CLASSES.items()}

# --- WINDOWS PATH ---
BASE_DIR      = Path(r"C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar")
COLORS_DIR    = BASE_DIR / "Classified" / "Colors"
FRAMES_DIR    = BASE_DIR / "Classified" / "Frames"
BANNERS_DIR   = BASE_DIR / "Classified" / "Banners"
SCREENSHOTS_DIR = BASE_DIR / "screenshots"
OUTPUT_DIR    = BASE_DIR / "synthetic_dataset"
IMAGES_DIR    = OUTPUT_DIR / "images"
LABELS_DIR    = OUTPUT_DIR / "labels"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RES = 640


# ==================== PHASE 2: FRAME CONFIGURATION ====================

FRAME_CONFIGS = {
    "standard_champion": {
        "file": "Classified/Frames/healthbar.png",
        "hp_x": 26, "hp_y": 5,  "hp_w": 107, "hp_h": 13,
        "mana_y": 19, "mana_h": 5, "has_mana": True,
        "banner_shape": "square", "banner_x": 4, "banner_y": 4,
        "level_x": 5, "level_y": 4, "font_size": 16,
    },
    "execute_threshold": {
        "file": "Classified/Frames/execute_treshold_health_frame.png",
        "hp_x": 30, "hp_y": 8,  "hp_w": 106, "hp_h": 12,
        "mana_y": 22, "mana_h": 5, "has_mana": True,
        "banner_shape": "rombhus", "banner_x": 8, "banner_y": 5,
        "level_x": 8, "level_y": 6, "font_size": 16,
    },
    "invulnerable": {
        "file": "Classified/Frames/frame_invulnerable.png",
        "hp_x": 27, "hp_y": 5,  "hp_w": 106, "hp_h": 13,
        "mana_y": 19, "mana_h": 5, "has_mana": True,
        "banner_shape": "shield", "banner_x": 4, "banner_y": 4,
        "level_x": 5, "level_y": 3, "font_size": 16,
    },
    "spellshield": {
        "file": "Classified/Frames/frame_spellshield.png",
        "hp_x": 27, "hp_y": 6,  "hp_w": 106, "hp_h": 12,
        "mana_y": 20, "mana_h": 5, "has_mana": True,
        "banner_shape": "rombhus", "banner_x": 4, "banner_y": 3,
        "level_x": 5, "level_y": 4, "font_size": 16,
    },
    "yuumi_special": {
        "file": "Classified/Frames/healthbar.png",
        "hp_x": 26, "hp_y": 5,  "hp_w": 104, "hp_h": 13,
        "mana_y": 19, "mana_h": 5, "has_mana": True,
        "overlay_file": "Classified/Frames/yummi_health_frame_cropped.png",
        "overlay_crop_w": 38, "overlay_x": -5, "overlay_y": -5,
        "banner_shape": "square", "banner_x": 4, "banner_y": 4,
        "level_x": 5, "level_y": 4, "font_size": 16,
    },
    "turret": {
        "file": "Classified/Frames/turret_frame.png",
        "hp_x": 11, "hp_y": 6,  "hp_w": 168, "hp_h": 15,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
    },
    "structure": {
        "file": "Classified/Frames/structure_frame.png",
        "hp_x": 5,  "hp_y": 4,  "hp_w": 142, "hp_h": 10,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
    },
    "epic_monster": {
        "file": "Classified/Frames/epic_monster.png",
        "hp_x": 12, "hp_y": 20, "hp_w": 168, "hp_h": 15,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
        "has_monster_number": True,
        "monster_font_size": 16,
        "monster_number_pivot_x": -2,
        "monster_number_offset_y": -23,
    },
    "jungle_monster_big": {
        "file": "Classified/Frames/jungle_monster_big.png",
        "hp_x": 4,  "hp_y": 21, "hp_w": 143, "hp_h": 12,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
        "has_monster_number": True,
        "monster_font_size": 16,
        "monster_number_pivot_x": -2,
        "monster_number_offset_y": -23,
    },
    # ── Minion frames ─────────────────────────────────────────────────────────
    # minion.png = standard lane minion (66×7, already RGBA)
    # No banner, no mana, no level, no tick marks — plain fill + frame only
    "minion": {
        "file": "Classified/Frames/minion.png",
        "hp_x": 2, "hp_y": 1, "hp_w": 62, "hp_h": 5,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
    },
    # jungle_mini_1.png = large mini jungle bar (147×7, already RGBA)
    "jungle_mini_1": {
        "file": "Classified/Frames/jungle_mini_1.png",
        "hp_x": 2, "hp_y": 1, "hp_w": 143, "hp_h": 5,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
    },
    # jungle_mini_2.png = small mini jungle bar (92×7, already RGBA)
    "jungle_mini_2": {
        "file": "Classified/Frames/jungle_mini_2.png",
        "hp_x": 2, "hp_y": 1, "hp_w": 88, "hp_h": 5,
        "mana_y": 0, "mana_h": 0, "has_mana": False,
    },
}

def _load_and_verify_configs():
    print("\n=== FRAME CONFIG VERIFICATION ===")
    valid = {}
    for name, cfg in FRAME_CONFIGS.items():
        p = BASE_DIR / cfg["file"]
        if p.exists():
            img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
            if img is not None:
                h, w = img.shape[:2]
                valid[name] = cfg
                print(f"  ✓ {name:22} | {w:3}x{h:3} | HP: ({cfg['hp_x']},{cfg['hp_y']}) {cfg['hp_w']}x{cfg['hp_h']}")
            else:
                print(f"  ✗ {name:22} | FAILED TO LOAD")
        else:
            print(f"  ✗ {name:22} | NOT FOUND")
    return valid

VALID_FRAMES = _load_and_verify_configs()


# ==================== PHASE 3: ASSET LOADER ====================

@dataclass
class ColorAsset:
    name: str
    image: np.ndarray
    color_type: str


class AssetLibrary:
    def __init__(self, colors_dir: Path):
        self.colors_dir  = colors_dir
        self.banners_dir = colors_dir.parent / "Banners"
        # BeaufortForLoL font — same path as original
        self.font_path   = colors_dir.parent / "Fonts" / "BeaufortForLoL-OTF" / "BeaufortForLoL-Bold.otf"
        self.hp_colors:      Dict[str, List[ColorAsset]] = {}
        self.special_effects: Dict[str, ColorAsset]      = {}
        self.banners:         Dict[str, ColorAsset]      = {}
        self.load_assets()

    def load_assets(self):
        if not self.colors_dir.exists():
            print(f"⚠ Colors dir missing: {self.colors_dir}")
            return

        # ── HP fills ──────────────────────────────────────────────────
        hp_files = {
            "player": ["green_character_hpcolor.png"],
            "ally":   ["ally_hpcolor.png"],
            "enemy":  ["enemy_hp_color.png"],
        }
        for team, files in hp_files.items():
            self.hp_colors[team] = []
            for fn in files:
                p = self.colors_dir / fn
                if p.exists():
                    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
                    if img is not None:
                        self.hp_colors[team].append(ColorAsset(fn, img, "hp"))

        # ── Special effects + resources (all keyed in special_effects) ─
        # Keeping the same dict as original so overlay_block() works identically
        special_files = {
            "shield":      "shield_normal.png",
            "grey_health": "grey_health.png",
            "spellshield": "shield_magic_purple.png",
            "kled":        "kled_extra_hpcolor.png",
            "mana":        "resource_mana_color.png",
            "energy":      "resource_energy_color.png",
            "fury":        "resource_fury_orange.png",
            "hp_gradient": "hp_green_gradient.png",
        }
        for key, fn in special_files.items():
            p = self.colors_dir / fn
            if p.exists():
                img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
                if img is not None:
                    self.special_effects[key] = ColorAsset(fn, img, key)

        # ── Banners ────────────────────────────────────────────────────
        if self.banners_dir.exists():
            banner_files = {
                "square_blue":  "square_blue.png",
                "square_red":   "square_red.png",
                "rombhus_blue": "rombhus_ally.png",
                "rombhus_red":  "rombhus_red.png",
                "shield_blue":  "shield_blue.png",
                "shield_red":   "shield_red.png",
            }
            for key, fn in banner_files.items():
                p = self.banners_dir / fn
                if p.exists():
                    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
                    if img is not None:
                        self.banners[key] = ColorAsset(fn, img, "banner")
            print(f"  ✓ Loaded {len(self.banners)} banner assets")
        else:
            print(f"  ⚠ Banners dir missing: {self.banners_dir}")

        print(f"  ✓ HP color teams: { {t: len(v) for t, v in self.hp_colors.items()} }")
        print(f"  ✓ Special effects: {list(self.special_effects.keys())}")

    def get_hp_color(self, team: str) -> Optional[ColorAsset]:
        lst = self.hp_colors.get(team, [])
        return random.choice(lst) if lst else None

    def get_special_effect(self, key: str) -> Optional[ColorAsset]:
        return self.special_effects.get(key)

    def get_banner(self, shape: str, team: str) -> Optional[ColorAsset]:
        color = "red" if team == "enemy" else "blue"
        return self.banners.get(f"{shape}_{color}")


assets = AssetLibrary(COLORS_DIR)


# ==================== PHASE 4: HEALTH BAR FACTORY ====================

class HealthBarFactory:
    def __init__(self, asset_lib: AssetLibrary, frame_configs: Dict):
        self.assets = asset_lib
        self.frames = frame_configs

    # ── Tick marks (ported verbatim from original) ─────────────────────
    def _draw_tick_marks(self, hp_strip: np.ndarray, full_hp_w: int,
                         max_hp: int, frame_name: str) -> np.ndarray:
        result = hp_strip.copy()
        h = result.shape[0]

        # Minion and jungle mini bars are smooth — no tick marks at all
        if frame_name in ("minion", "jungle_mini_1", "jungle_mini_2"):
            return result

        if frame_name == "turret":
            num_segments, gap_w = 5, 2
            for i in range(1, num_segments):
                x_pos = int(i * full_hp_w / num_segments)
                if 0 <= x_pos - gap_w < result.shape[1]:
                    result[:, max(0, x_pos - gap_w):min(x_pos, result.shape[1]), 3] = 0
            return result

        if frame_name == "structure" or max_hp <= 0:
            return result

        if frame_name in ("epic_monster", "jungle_monster_big"):
            px_per_1k = (full_hp_w / max_hp) * 1000
            cur = px_per_1k
            while cur < result.shape[1]:
                cv2.line(result, (int(cur), 0), (int(cur), h), (0, 0, 0, 255), 1)
                cur += px_per_1k
        else:
            px_per_100 = (full_hp_w / max_hp) * 100
            cur, hp_count = px_per_100, 100
            tick_h = max(1, int(h * 0.4))
            while cur < result.shape[1]:
                x = int(cur)
                if hp_count % 1000 == 0:
                    cv2.line(result, (x, 0), (x, h), (0, 0, 0, 255), 1)
                else:
                    cv2.line(result, (x, 0), (x, tick_h), (0, 0, 0, 255), 1)
                cur += px_per_100
                hp_count += 100
        return result

    # ── Gradient-preserving proportional strip resize ──────────────────
    def _overlay_block(self, strip: np.ndarray, width: int, start_x: int,
                       asset_name: str, hp_w: int, hp_h: int,
                       is_team_color: bool = False) -> Tuple[np.ndarray, int]:
        """
        Faithfully ported from the original notebook.
        Crops the source asset proportionally so gradients look natural,
        then alpha-blends into `strip` at `start_x`.
        """
        if width <= 0 or start_x >= hp_w:
            return strip, start_x

        if is_team_color:
            asset = self.assets.get_hp_color(asset_name)
        else:
            asset = self.assets.get_special_effect(asset_name)

        if asset is None:
            return strip, start_x

        w_clamped = min(width, hp_w - start_x)
        if w_clamped <= 0:
            return strip, start_x

        img = asset.image.copy()

        # Auto-crop dead (dark/transparent) padding from the source strip
        if img.shape[2] == 4:
            gray = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2GRAY)
            mask = ((img[:, :, 3] > 20) & (gray > 20)).astype(np.uint8)
        else:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            mask = (gray > 20).astype(np.uint8)
        coords = cv2.findNonZero(mask)
        if coords is not None:
            cx, cy, cw, ch = cv2.boundingRect(coords)
            img = img[:, cx: cx + cw]

        # Proportional crop so gradient fills naturally
        native_w = img.shape[1]
        crop_w = max(1, min(native_w, int(native_w * (width / hp_w))))
        cropped = img[:, :crop_w]

        layer = cv2.resize(cropped, (w_clamped, hp_h), interpolation=cv2.INTER_LINEAR)

        if layer.shape[2] == 4:
            alpha = layer[:, :, 3:4] / 255.0
            strip[:, start_x: start_x + w_clamped, :3] = (
                layer[:, :, :3] * alpha +
                strip[:, start_x: start_x + w_clamped, :3] * (1.0 - alpha)
            ).astype(np.uint8)
            strip[:, start_x: start_x + w_clamped, 3] = np.maximum(
                strip[:, start_x: start_x + w_clamped, 3], layer[:, :, 3]
            )
        else:
            strip[:, start_x: start_x + w_clamped, :3] = layer[:, :, :3]
            strip[:, start_x: start_x + w_clamped, 3] = 255

        return strip, start_x + w_clamped

    # ── Resource bar crop helper (verbatim from original) ──────────────
    def _crop_color_to_percentage(self, color_strip: np.ndarray,
                                   percentage: float) -> np.ndarray:
        _, crop_w = color_strip.shape[:2]
        target_w = int(crop_w * max(0, min(100, percentage)) / 100)
        return color_strip[:, :max(1, target_w)]

    # ── Alpha blend (verbatim from original) ───────────────────────────
    def _blend_alpha(self, dst_roi: np.ndarray, src: np.ndarray, max_width: int):
        if src.shape[0] == 0 or src.shape[1] == 0:
            return
        h_dst, w_dst = dst_roi.shape[:2]
        src_r = cv2.resize(src, (min(src.shape[1], w_dst), h_dst),
                           interpolation=cv2.INTER_NEAREST)
        if src_r.shape[2] == 4:
            alpha = src_r[:, :, 3:4] / 255.0
            for c in range(3):
                dst_roi[:, :src_r.shape[1], c] = (
                    src_r[:, :, c] * alpha[:, :, 0] +
                    dst_roi[:, :src_r.shape[1], c] * (1 - alpha[:, :, 0])
                ).astype(np.uint8)
            dst_roi[:, :src_r.shape[1], 3] = np.maximum(
                dst_roi[:, :src_r.shape[1], 3], src_r[:, :, 3]
            )
        else:
            dst_roi[:, :src_r.shape[1], :3] = src_r

    # ── TTF level/number text (verbatim from original) ─────────────────
    def _draw_level_text(self, canvas: np.ndarray, text: str,
                         x: int, y: int, font_size: int) -> np.ndarray:
        img_rgb  = cv2.cvtColor(canvas, cv2.COLOR_BGRA2RGBA)
        pil_img  = Image.fromarray(img_rgb)
        draw     = ImageDraw.Draw(pil_img)
        try:
            font = ImageFont.truetype(str(self.assets.font_path), font_size)
        except Exception:
            font = ImageFont.load_default()
        # Drop shadow then white text
        draw.text((x + 1, y + 1), text, font=font, fill=(0, 0, 0, 255))
        draw.text((x,     y    ), text, font=font, fill=(255, 255, 255, 255))
        return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGBA2BGRA)

    # ── Main factory method ────────────────────────────────────────────
    def create_healthbar(
        self,
        team: str,
        frame_name: str,
        hp_percentage:   Optional[float] = None,
        mana_percentage: Optional[float] = None,
        shield_pct:      float = 0.0,
        spellshield_pct: float = 0.0,
        grey_hp_pct:     float = 0.0,
        kled_hp_pct:     float = 0.0,
    ) -> Tuple[Optional[np.ndarray], Optional[Dict]]:

        if frame_name not in self.frames:
            print(f"⚠ frame '{frame_name}' not in VALID_FRAMES")
            return None, None

        cfg = self.frames[frame_name]
        frame_img = cv2.imread(str(BASE_DIR / cfg["file"]), cv2.IMREAD_UNCHANGED)
        if frame_img is None:
            print(f"⚠ cv2 failed: {BASE_DIR / cfg['file']}")
            return None, None
        frame_img = frame_img.copy()

        # Resolved HP color team (monsters / structures always use ally blue)
        is_non_champion = frame_name in ("turret", "structure", "epic_monster", "jungle_monster_big")
        resolved_team = "ally" if (is_non_champion and team in ("player", "ally")) else team

        # Strip champion-only effects for non-champion frames
        if is_non_champion:
            shield_pct = spellshield_pct = grey_hp_pct = kled_hp_pct = 0.0

        # Max HP determines tick-mark spacing
        if frame_name == "epic_monster":
            max_hp = random.randint(2500, 30000)
        elif frame_name == "jungle_monster_big":
            max_hp = random.randint(500, 3000)
        else:
            max_hp = random.randint(600, 4000)

        hp_pct    = hp_percentage   if hp_percentage   is not None else random.uniform(10.0, 100.0)
        hp_pct    = max(0.0, min(100.0, hp_pct))
        mana_pct  = mana_percentage if mana_percentage is not None else random.uniform(30.0, 100.0)
        current_hp = int(round(max_hp * hp_pct / 100.0))

        # ── Canvas ──────────────────────────────────────────────────────
        pad = 30
        canvas_h = frame_img.shape[0] + pad * 2
        canvas_w = frame_img.shape[1] + pad * 2
        result   = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
        bx, by   = pad, pad

        # ── 1. Dark HP background ────────────────────────────────────
        empty_bg  = np.full((cfg["hp_h"], cfg["hp_w"], 4), (30, 30, 30, 255), dtype=np.uint8)
        empty_roi = result[by + cfg["hp_y"]: by + cfg["hp_y"] + cfg["hp_h"],
                           bx + cfg["hp_x"]: bx + cfg["hp_x"] + cfg["hp_w"]]
        self._blend_alpha(empty_roi, empty_bg, cfg["hp_w"])

        # ── 2. Build HP fill strip (A→D sequential blocks) ──────────
        target_hp_w = int(round(cfg["hp_w"] * hp_pct / 100.0)) if hp_pct < 99.5 else cfg["hp_w"]
        combined    = np.zeros((cfg["hp_h"], cfg["hp_w"], 4), dtype=np.uint8)
        current_x   = 0

        # A. Base HP colour
        combined, current_x = self._overlay_block(
            combined, target_hp_w, current_x,
            resolved_team, cfg["hp_w"], cfg["hp_h"], is_team_color=True
        )

        # B. Kled secondary courage bar (appended right after HP)
        if kled_hp_pct > 0.0:
            kled_w = int(round(cfg["hp_w"] * kled_hp_pct / 100.0))
            combined, current_x = self._overlay_block(
                combined, kled_w, current_x,
                "kled", cfg["hp_w"], cfg["hp_h"]
            )

        base_hp_end_x = current_x

        # C. Grey health (damage-absorb shield, fills remaining empty space)
        if grey_hp_pct > 0.0 and base_hp_end_x < cfg["hp_w"]:
            grey_w = int(round(cfg["hp_w"] * grey_hp_pct / 100.0))
            self._overlay_block(
                combined, grey_w, base_hp_end_x,
                "grey_health", cfg["hp_w"], cfg["hp_h"]
            )  # note: does NOT advance current_x — grey sits independently

        # D. Shield overlays (append after current HP end)
        shield_w = int(round(cfg["hp_w"] * shield_pct / 100.0))
        combined, current_x = self._overlay_block(
            combined, shield_w, base_hp_end_x,
            "shield", cfg["hp_w"], cfg["hp_h"]
        )

        ss_w = int(round(cfg["hp_w"] * spellshield_pct / 100.0))
        combined, current_x = self._overlay_block(
            combined, ss_w, current_x,
            "spellshield", cfg["hp_w"], cfg["hp_h"]
        )

        # ── 3. Render HP strip + tick marks onto canvas ──────────────
        total_fill_w = min(cfg["hp_w"], max(base_hp_end_x, current_x))
        if total_fill_w > 0:
            filled   = combined[:, :total_fill_w]
            with_ticks = self._draw_tick_marks(filled, cfg["hp_w"], max_hp, frame_name)
            hp_roi   = result[by + cfg["hp_y"]: by + cfg["hp_y"] + cfg["hp_h"],
                              bx + cfg["hp_x"]: bx + cfg["hp_x"] + total_fill_w]
            self._blend_alpha(hp_roi, with_ticks, total_fill_w)

        # ── 4. Punch alpha hole in frame over HP slot ────────────────
        # (Prevents dark artifact behind low-HP bars — exact original fix)
        frame_img[cfg["hp_y"]: cfg["hp_y"] + cfg["hp_h"],
                  cfg["hp_x"]: cfg["hp_x"] + cfg["hp_w"], 3] = 0

        # ── 5. Resource bar (mana / energy / fury) ───────────────────
        if cfg["has_mana"] and cfg["mana_h"] > 0:
            resource_type  = random.choice(["mana", "energy", "fury"])
            resource_asset = self.assets.get_special_effect(resource_type)
            if resource_asset is not None:
                mana_cropped  = self._crop_color_to_percentage(resource_asset.image, mana_pct)
                actual_mana_w = mana_cropped.shape[1]
                mana_roi = result[by + cfg["mana_y"]: by + cfg["mana_y"] + cfg["mana_h"],
                                  bx + cfg["hp_x"]:   bx + cfg["hp_x"]  + actual_mana_w]
                self._blend_alpha(mana_roi, mana_cropped, actual_mana_w)
                # Punch frame hole over mana slot too
                frame_img[cfg["mana_y"]: cfg["mana_y"] + cfg["mana_h"],
                          cfg["hp_x"]:   cfg["hp_x"]  + actual_mana_w, 3] = 0

        # ── 6. Banner (level badge) ───────────────────────────────────
        if "banner_shape" in cfg:
            banner = self.assets.get_banner(cfg["banner_shape"], team)
            if banner is not None:
                bx2 = bx + cfg.get("banner_x", 0)
                by2 = by + cfg.get("banner_y", 0)
                if bx2 >= 0 and by2 >= 0:
                    b_roi = result[by2: by2 + banner.image.shape[0],
                                   bx2: bx2 + banner.image.shape[1]]
                    self._blend_alpha(b_roi, banner.image, banner.image.shape[1])

        # ── 7. Frame chrome ───────────────────────────────────────────
        frame_roi = result[by: by + frame_img.shape[0], bx: bx + frame_img.shape[1]]
        self._blend_alpha(frame_roi, frame_img, frame_img.shape[1])

        # ── 8. Level number text ──────────────────────────────────────
        if "level_x" in cfg:
            lvl   = str(random.randint(1, 18))
            lvl_x = bx + cfg["level_x"] + (3 if len(lvl) == 1 else 0)
            lvl_y = by + cfg["level_y"]
            result = self._draw_level_text(result, lvl, lvl_x, lvl_y, cfg["font_size"])

        # ── 9. Monster number (current HP shown above bar) ────────────
        if cfg.get("has_monster_number", False):
            m_num   = str(current_hp)
            font_sz = cfg.get("monster_font_size", 14)
            hp_cx   = bx + cfg["hp_x"] + cfg["hp_w"] // 2
            pivot_x = hp_cx + cfg.get("monster_number_pivot_x", 0)
            approx_w = (font_sz // 2) * len(m_num)
            num_x   = pivot_x - approx_w // 2
            num_y   = by + cfg["hp_y"] + cfg.get("monster_number_offset_y", -12)
            result  = self._draw_level_text(result, m_num, num_x, num_y, font_sz)

        # ── 10. Yuumi overlay ─────────────────────────────────────────
        if "overlay_file" in cfg:
            ov_path = BASE_DIR / cfg["overlay_file"]
            if ov_path.exists():
                ov = cv2.imread(str(ov_path), cv2.IMREAD_UNCHANGED)
                if ov is not None:
                    ov = ov[:, :cfg.get("overlay_crop_w", ov.shape[1])]
                    ox = bx + cfg.get("overlay_x", 0)
                    oy = by + cfg.get("overlay_y", 0)
                    if ox >= 0 and oy >= 0:
                        ov_roi = result[oy: oy + ov.shape[0], ox: ox + ov.shape[1]]
                        self._blend_alpha(ov_roi, ov, ov.shape[1])

        # ── 11. Auto-crop to content ──────────────────────────────────
        non_zero = cv2.findNonZero(result[:, :, 3])
        if non_zero is None:
            return None, None
        x, y, w, h = cv2.boundingRect(non_zero)
        cropped = result[y: y + h, x: x + w]

        metadata = {
            "team":   team,
            "frame":  frame_name,
            "hp_pct": hp_pct,
            "hp_x":   (bx + cfg["hp_x"]) - x,
            "hp_y":   (by + cfg["hp_y"]) - y,
            "hp_w":   cfg["hp_w"],
            "hp_h":   cfg["hp_h"],
            "width":  w,
            "height": h,
        }
        return cropped, metadata


factory = HealthBarFactory(assets, VALID_FRAMES)


# ==================== PHASE 5: COMPOSITOR ====================

# Spawn table — mirrors which (team, frame) produce each YOLO class
CLASS_SPAWN_PARAMS: Dict[int, List[Tuple[str, str]]] = {
    0:  [("player", "standard_champion"), ("player", "execute_threshold"),
         ("player", "invulnerable"),      ("player", "spellshield")],
    1:  [("player", "yuumi_special")],
    2:  [("ally",   "standard_champion"), ("ally",   "execute_threshold"),
         ("ally",   "invulnerable"),      ("ally",   "spellshield")],
    3:  [("ally",   "yuumi_special")],
    4:  [("enemy",  "standard_champion"), ("enemy",  "execute_threshold"),
         ("enemy",  "invulnerable"),      ("enemy",  "spellshield")],
    5:  [("enemy",  "yuumi_special")],
    6:  [("ally",   "turret")],
    7:  [("enemy",  "turret")],
    8:  [("ally",   "structure")],
    9:  [("enemy",  "structure")],
    10: [("ally",   "jungle_monster_big")],
    11: [("enemy",  "epic_monster")],
    12: [("ally",   "minion")],
    13: [("enemy",  "minion")],
    14: [("ally",   "jungle_mini_1"), ("ally",  "jungle_mini_2"),
         ("enemy",  "jungle_mini_1"), ("enemy", "jungle_mini_2")],
}

def _resolve_yolo_class(frame_name: str, team: str) -> int:
    if frame_name == "turret":
        name = "Ally Turret"    if team in ("player", "ally") else "Enemy Turret"
    elif frame_name == "structure":
        name = "Ally Structure" if team in ("player", "ally") else "Enemy Structure"
    elif frame_name == "epic_monster":
        name = "Epic Monster"
    elif frame_name == "jungle_monster_big":
        name = "Jungle Monster"
    elif frame_name in ("jungle_mini_1", "jungle_mini_2"):
        name = "Mini Jungle Monster"
    elif frame_name == "minion":
        name = "Ally Minion" if team in ("player", "ally") else "Enemy Minion"
    elif frame_name == "yuumi_special":
        name = {"player": "Player+Yuumi", "ally": "Ally+Yuumi"}.get(team, "Enemy+Yuumi")
    else:
        name = {"player": "Player", "ally": "Ally"}.get(team, "Enemy")
    return CLASS_TO_ID.get(name, 0)


class ScreenshotCompositor:
    def __init__(self, screenshots_dir: Path, output_dir: Path):
        self.image_output = output_dir / "images"
        self.label_output = output_dir / "labels"
        self.screenshots  = []
        for ext in ("*.png", "*.PNG", "*.jpg", "*.JPG", "*.jpeg", "*.JPEG"):
            self.screenshots.extend(list(screenshots_dir.glob(ext)))

    def _apply_perspective_warp(self, img, labels):
        H, W = img.shape[:2]
        sx, sy = int(W * 0.08), int(H * 0.08)
        src = np.float32([[0,0],[W,0],[W,H],[0,H]])
        dst = np.float32([
            [random.randint(0,sx), random.randint(0,sy)],
            [W-random.randint(0,sx), random.randint(0,sy)],
            [W-random.randint(0,sx), H-random.randint(0,sy)],
            [random.randint(0,sx), H-random.randint(0,sy)],
        ])
        M = cv2.getPerspectiveTransform(src, dst)
        warped = cv2.warpPerspective(img, M, (W,H), borderMode=cv2.BORDER_REPLICATE)
        new_labels = []
        for cls_id, cx, cy, nw, nh in labels:
            xc, yc = cx*W, cy*H
            bw, bh = nw*W, nh*H
            corners = np.float32([[[xc-bw/2,yc-bh/2]],[[xc+bw/2,yc-bh/2]],
                                   [[xc+bw/2,yc+bh/2]],[[xc-bw/2,yc+bh/2]]])
            wc = cv2.perspectiveTransform(corners, M)
            xs, ys = wc[:,0,0], wc[:,0,1]
            nx1,nx2 = np.clip([xs.min(),xs.max()],0,W)
            ny1,ny2 = np.clip([ys.min(),ys.max()],0,H)
            nw2,nh2 = nx2-nx1, ny2-ny1
            if nw2>2 and nh2>2:
                new_labels.append((cls_id,(nx1+nw2/2)/W,(ny1+nh2/2)/H,nw2/W,nh2/H))
        return warped, new_labels

    def _apply_camera_augmentations(self, img):
        r = img.copy()
        if random.random() < 0.6:
            r = cv2.convertScaleAbs(r, alpha=random.uniform(0.6,1.4), beta=random.randint(-40,40))
        if random.random() < 0.3:
            sz = random.randint(3,9)
            k  = np.zeros((sz,sz)); k[sz//2,:] = 1/sz
            r  = cv2.filter2D(r, -1, k)
        elif random.random() < 0.4:
            k = random.choice([3,5,7])
            r = cv2.GaussianBlur(r,(k,k),0)
        if random.random() < 0.5:
            n = random.randint(10,40)
            r = np.clip(r.astype(np.int16)+np.random.randint(-n,n,r.shape),0,255).astype(np.uint8)
        if random.random() < 0.7:
            _,enc = cv2.imencode('.jpg',r,[int(cv2.IMWRITE_JPEG_QUALITY),random.randint(15,60)])
            r = cv2.imdecode(enc,1)
        return r

    def compose_scene(self, entities: List[Tuple[np.ndarray, Dict]], sample_id: int) -> bool:
        if not self.screenshots or not entities:
            return False
        try:
            bg = cv2.imread(str(random.choice(self.screenshots)), cv2.IMREAD_COLOR)
            if bg is None:
                return False
            bg = cv2.resize(bg, (TARGET_RES, TARGET_RES), interpolation=cv2.INTER_AREA)
            W, H = TARGET_RES, TARGET_RES
            raw_labels = []

            # One scene_scale applied uniformly to every sprite's native pixel size.
            scene_scale = random.uniform(0.3, 1)

            for hb, meta in entities:
                hb_h, hb_w = hb.shape[:2]
                new_w = max(1, round(hb_w * scene_scale))
                new_h = max(1, round(hb_h * scene_scale))
                scaled = cv2.resize(hb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

                s_hp_w = meta["hp_w"] * scene_scale
                s_hp_h = meta["hp_h"] * scene_scale
                s_hp_x = meta["hp_x"] * scene_scale
                s_hp_y = meta["hp_y"] * scene_scale

                # ── Placement ────────────────────────────────────────────
                # 25 % chance: allow the bar to hang off any edge (partial crop)
                # 75 % chance: place fully inside with a small margin
                if random.random() < 0.25:
                    # Allow placement anywhere from -new_w+1 to W-1 (same for Y)
                    # so the bar can bleed off any edge but at least 1px stays on canvas
                    px = random.randint(-new_w + 1, W - 1)
                    py = random.randint(-new_h + 1, H - 1)
                else:
                    margin = int(TARGET_RES * 0.05)   # tiny margin, bars can still overlap
                    px = random.randint(-margin, W - new_w + margin)
                    py = random.randint(-margin, H - new_h + margin)

                # ── Clip to canvas and paste only the visible portion ────
                # Source slice (into scaled bar)
                src_x0 = max(0, -px);          src_y0 = max(0, -py)
                src_x1 = min(new_w, W - px);   src_y1 = min(new_h, H - py)
                # Destination slice (into bg)
                dst_x0 = max(0, px);            dst_y0 = max(0, py)
                dst_x1 = dst_x0 + (src_x1 - src_x0)
                dst_y1 = dst_y0 + (src_y1 - src_y0)

                vis_w = src_x1 - src_x0
                vis_h = src_y1 - src_y0
                if vis_w <= 0 or vis_h <= 0:
                    continue   # entirely off canvas

                patch     = scaled[src_y0:src_y1, src_x0:src_x1]
                alpha     = patch[:, :, 3:4] / 255.0
                bg_roi    = bg[dst_y0:dst_y1, dst_x0:dst_x1]
                bg[dst_y0:dst_y1, dst_x0:dst_x1] = (
                    patch[:, :, :3] * alpha + bg_roi * (1.0 - alpha)
                ).astype(np.uint8)

                # ── YOLO label — HP fill box clipped to canvas ───────────
                # Absolute HP fill rect on canvas (may be partially outside)
                hp_abs_x1 = px + s_hp_x
                hp_abs_y1 = py + s_hp_y
                hp_abs_x2 = hp_abs_x1 + s_hp_w
                hp_abs_y2 = hp_abs_y1 + s_hp_h

                # Clip to canvas
                hp_vis_x1 = max(0.0, hp_abs_x1)
                hp_vis_y1 = max(0.0, hp_abs_y1)
                hp_vis_x2 = min(float(W), hp_abs_x2)
                hp_vis_y2 = min(float(H), hp_abs_y2)

                hp_vis_w = hp_vis_x2 - hp_vis_x1
                hp_vis_h = hp_vis_y2 - hp_vis_y1

                # Skip label if less than 40 % of the HP fill is visible
                if hp_vis_w <= 0 or hp_vis_h <= 0:
                    continue
                if (hp_vis_w * hp_vis_h) < 0.40 * (s_hp_w * s_hp_h):
                    continue

                cls_id = _resolve_yolo_class(meta["frame"], meta["team"])
                cx = (hp_vis_x1 + hp_vis_w / 2) / W
                cy = (hp_vis_y1 + hp_vis_h / 2) / H
                raw_labels.append((cls_id, cx, cy, hp_vis_w / W, hp_vis_h / H))

            if not raw_labels:
                return False

            if random.random() > 0.30:
                bg, raw_labels = self._apply_perspective_warp(bg, raw_labels)

            final   = self._apply_camera_augmentations(bg)
            strs    = [f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c,x,y,w,h in raw_labels]
            cv2.imwrite(str(self.image_output / f"scene_{sample_id:05d}.jpg"), final)
            with open(self.label_output / f"scene_{sample_id:05d}.txt","w") as f:
                f.write("\n".join(strs)+"\n")
            return True
        except Exception as e:
            print(f"⚠ ERROR scene {sample_id}: {e}")
            return False


compositor = ScreenshotCompositor(SCREENSHOTS_DIR, OUTPUT_DIR)


# ==================== PHASE 6: GENERATION PIPELINE ====================

def _random_effects_for_champion(frame_name: str):
    """Return (shield_pct, spellshield_pct, grey_hp_pct, kled_hp_pct)"""
    is_champ = frame_name not in (
        "turret", "structure", "epic_monster", "jungle_monster_big",
        "minion", "jungle_mini_1", "jungle_mini_2",
    )
    if not is_champ:
        return 0.0, 0.0, 0.0, 0.0
    shield_pct      = random.uniform(10, 35) if random.random() < 0.10 else 0.0
    spellshield_pct = random.uniform(15, 40) if random.random() < 0.04 else 0.0
    grey_hp_pct     = random.uniform(10, 25) if random.random() < 0.07 else 0.0
    kled_hp_pct     = random.uniform(20, 80) if random.random() < 0.03 else 0.0
    return shield_pct, spellshield_pct, grey_hp_pct, kled_hp_pct


def _spawn_entity(team: str, frame_name: str) -> Tuple[Optional[np.ndarray], Optional[Dict]]:
    s, ss, g, k = _random_effects_for_champion(frame_name)
    return factory.create_healthbar(
        team=team, frame_name=frame_name,
        shield_pct=s, spellshield_pct=ss,
        grey_hp_pct=g, kled_hp_pct=k,
    )


def _pick_scene_density() -> int:
    """
    Variable scene density reflecting real gameplay variety:
      20% quiet   —  2- 5 bars  (solo laning, recalled to base, jungle)
      50% normal  —  6-15 bars  (lane with minions, small skirmish)
      30% chaotic — 16-30 bars  (teamfight, lane+jungle combined)
    """
    roll = random.random()
    if roll < 0.20:
        return random.randint(2, 5)
    elif roll < 0.70:
        return random.randint(6, 15)
    else:
        return random.randint(16, 30)


def generate_synthetic_dataset(num_scenes: int):
    print(f"\n{'='*60}")
    print("PRE-FLIGHT DIAGNOSTICS")
    print(f"{'='*60}")

    if not VALID_FRAMES:
        print(f"❌ FATAL: No frames found in {FRAMES_DIR}"); return 0, 0
    print(f"✓ {len(VALID_FRAMES)} frames loaded.")

    if not compositor.screenshots:
        print(f"❌ FATAL: No backgrounds in {SCREENSHOTS_DIR}"); return 0, 0
    print(f"✓ {len(compositor.screenshots)} backgrounds loaded.")

    for fname in ("minion", "jungle_mini_1", "jungle_mini_2"):
        if fname not in VALID_FRAMES:
            print(f"  ⚠ '{fname}' not found — class will be skipped.")

    successful, failed = 0, 0

    # Global counters — drive class AND frame balance across the whole run
    class_counts = {i: 0 for i in range(len(YOLO_CLASSES))}
    frame_counts: Dict[str, int] = {}   # "team|frame_name" → global count

    # ── Per-class max per scene ───────────────────────────────────────────────
    # Reflects realistic in-game co-occurrence maximums.
    CLASS_MAX_PER_SCENE = {
        0:  2,    # Player
        1:  1,    # Player+Yuumi
        2:  3,    # Ally
        3:  2,    # Ally+Yuumi
        4:  3,    # Enemy
        5:  2,    # Enemy+Yuumi
        6:  2,    # Ally Turret
        7:  2,    # Enemy Turret
        8:  1,    # Ally Structure
        9:  1,    # Enemy Structure
        10: 2,    # Jungle Monster
        11: 1,    # Epic Monster
        12: 12,   # Ally Minion  (full wave)
        13: 12,   # Enemy Minion (full wave)
        14: 6,    # Mini Jungle Monster
    }

    # Only spawn classes that have at least one valid frame
    available_classes = [
        c for c, params in CLASS_SPAWN_PARAMS.items()
        if any(fr in VALID_FRAMES for _, fr in params)
    ]

    # Initialise frame_counts for every valid (team, frame) option
    for cls_params in CLASS_SPAWN_PARAMS.values():
        for team, frame in cls_params:
            if frame in VALID_FRAMES:
                key = f"{team}|{frame}"
                frame_counts.setdefault(key, 0)

    print(f"\n{'='*60}")
    print(f"GENERATING {num_scenes} SCENES @ {TARGET_RES}×{TARGET_RES}")
    print(f"{'='*60}")

    for i in range(num_scenes):
        scene_entities: List[Tuple[np.ndarray, Dict]] = []
        scene_cls_counts = {c: 0 for c in available_classes}
        n_entities = _pick_scene_density()

        for _ in range(n_entities):
            # Candidates: classes not yet at their per-scene cap
            candidates = [
                c for c in available_classes
                if scene_cls_counts[c] < CLASS_MAX_PER_SCENE[c]
            ]
            if not candidates:
                break

            # Pick globally least-seen class
            target_cls = min(candidates, key=class_counts.get)

            # Pick globally least-seen (team, frame) option for that class
            options = [
                (t, fr) for t, fr in CLASS_SPAWN_PARAMS[target_cls]
                if fr in VALID_FRAMES
            ]
            if not options:
                continue
            team, frame = min(options, key=lambda tf: frame_counts.get(f"{tf[0]}|{tf[1]}", 0))

            hb, meta = _spawn_entity(team, frame)
            if hb is not None:
                scene_entities.append((hb, meta))
                class_counts[target_cls]            += 1
                scene_cls_counts[target_cls]        += 1
                frame_counts[f"{team}|{frame}"]     += 1

        if compositor.compose_scene(scene_entities, i):
            successful += 1
        else:
            failed += 1

        if (i + 1) % max(1, num_scenes // 10) == 0:
            print(f"  [{(i+1)/num_scenes*100:3.0f}%] {i+1:5d}/{num_scenes} scenes …")

    print(f"\n{'='*60}")
    print("COMPLETE")
    print(f"  ✓ Successful : {successful}")
    print(f"  ✗ Failed     : {failed}")

    print(f"\nClass distribution:")
    for cid, cnt in class_counts.items():
        print(f"  [{cid:2d}] {YOLO_CLASSES[cid]:<22}: {cnt}")

    print(f"\nFrame distribution:")
    for key, cnt in sorted(frame_counts.items(), key=lambda x: x[1]):
        print(f"  {key:<35}: {cnt}")

    print(f"{'='*60}\n")
    return successful, failed


if __name__ == "__main__":
    success, fail = generate_synthetic_dataset(num_scenes=200000)


=== FRAME CONFIG VERIFICATION ===
  ✓ standard_champion      | 136x 29 | HP: (26,5) 107x13
  ✓ execute_threshold      | 140x 35 | HP: (30,8) 106x12
  ✓ invulnerable           | 136x 33 | HP: (27,5) 106x13
  ✓ spellshield            | 136x 31 | HP: (27,6) 106x12
  ✓ yuumi_special          | 136x 29 | HP: (26,5) 104x13
  ✓ turret                 | 192x 34 | HP: (11,6) 168x15
  ✓ structure              | 151x 23 | HP: (5,4) 142x10
  ✓ epic_monster           | 192x 48 | HP: (12,20) 168x15
  ✓ jungle_monster_big     | 151x 43 | HP: (4,21) 143x12
  ✓ minion                 |  66x  7 | HP: (2,1) 62x5
  ✓ jungle_mini_1          | 147x  7 | HP: (2,1) 143x5
  ✓ jungle_mini_2          |  92x  7 | HP: (2,1) 88x5
  ✓ Loaded 6 banner assets
  ✓ HP color teams: {'player': 1, 'ally': 1, 'enemy': 1}
  ✓ Special effects: ['shield', 'grey_health', 'spellshield', 'kled', 'mana', 'energy', 'fury', 'hp_gradient']

PRE-FLIGHT DIAGNOSTICS
✓ 12 frames loaded.
✓ 1784 backgrounds loaded.

GENERATING 200000 SCEN

In [ ]:
import os
import shutil
import random
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================

BASE_DIR   = Path(r"C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar")
SOURCE_DIR = BASE_DIR / "synthetic_dataset"
SRC_IMAGES = SOURCE_DIR / "images"
SRC_LABELS = SOURCE_DIR / "labels"

OUTPUT_DIR = BASE_DIR / "yolo_training_data"

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

YOLO_CLASSES = {
    0:  "Player",
    1:  "Player+Yuumi",
    2:  "Ally",
    3:  "Ally+Yuumi",
    4:  "Enemy",
    5:  "Enemy+Yuumi",
    6:  "Ally Turret",
    7:  "Enemy Turret",
    8:  "Ally Structure",
    9:  "Enemy Structure",
    10: "Jungle Monster",
    11: "Epic Monster",
    12: "Ally Minion",
    13: "Enemy Minion",
    14: "Mini Jungle Monster",
}

# ==========================================
# EXECUTION
# ==========================================
def prepare_dataset():
    print("========================================")
    print(" YOLO DATASET PREPARATION PIPELINE")
    print("========================================\n")

    print("Scanning source directories...")
    if not SRC_IMAGES.exists() or not SRC_LABELS.exists():
        print(f"❌ Error: Source images or labels directory not found at {SOURCE_DIR}")
        return

    image_files = [f for f in os.listdir(SRC_IMAGES) if f.endswith('.jpg')]

    valid_pairs   = []
    missing_labels = 0

    for img_file in image_files:
        base_name = os.path.splitext(img_file)[0]
        lbl_file  = base_name + ".txt"
        if (SRC_LABELS / lbl_file).exists():
            valid_pairs.append(base_name)
        else:
            missing_labels += 1

    total_valid = len(valid_pairs)
    print(f"✓ Found {len(image_files)} images.")
    print(f"✓ Found {total_valid} complete image-label pairs.")
    if missing_labels > 0:
        print(f"⚠ Warning: {missing_labels} images were missing labels and will be ignored.")

    if total_valid == 0:
        print("❌ Error: No valid data to split. Exiting.")
        return

    print("\nShuffling data and calculating splits...")
    random.shuffle(valid_pairs)

    train_count = int(total_valid * TRAIN_RATIO)
    val_count   = int(total_valid * VAL_RATIO)

    train_data = valid_pairs[:train_count]
    val_data   = valid_pairs[train_count:train_count + val_count]
    test_data  = valid_pairs[train_count + val_count:]

    print(f"  - Training Set:   {len(train_data)} images ({TRAIN_RATIO*100:.0f}%)")
    print(f"  - Validation Set: {len(val_data)} images ({VAL_RATIO*100:.0f}%)")
    print(f"  - Testing Set:    {len(test_data)} images ({TEST_RATIO*100:.0f}%)")

    print("\nCreating YOLO directory structure...")
    if OUTPUT_DIR.exists():
        print("⚠ Output directory already exists. Clearing old data...")
        try:
            shutil.rmtree(OUTPUT_DIR)
        except Exception as e:
            print(f"❌ Error clearing directory: {e}. Please delete {OUTPUT_DIR} manually.")
            return

    for split in ['train', 'val', 'test']:
        (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

    print("Copying files to splits (this may take a moment)...")

    def copy_files(file_list, split_name):
        for base_name in file_list:
            shutil.copy(SRC_IMAGES / f"{base_name}.jpg",
                        OUTPUT_DIR / 'images' / split_name / f"{base_name}.jpg")
            shutil.copy(SRC_LABELS / f"{base_name}.txt",
                        OUTPUT_DIR / 'labels' / split_name / f"{base_name}.txt")

    copy_files(train_data, 'train')
    copy_files(val_data,   'val')
    copy_files(test_data,  'test')
    print("✓ All files copied successfully.")

    print("\nGenerating data.yaml...")
    yaml_path        = OUTPUT_DIR / "data.yaml"
    safe_output_path = OUTPUT_DIR.absolute().as_posix()

    yaml_content = f"""# YOLOv8 Dataset Configuration — 15 classes
path: {safe_output_path}
train: images/train
val:   images/val
test:  images/test

nc: {len(YOLO_CLASSES)}

names:
"""
    for cls_id in sorted(YOLO_CLASSES.keys()):
        yaml_content += f"  {cls_id}: {YOLO_CLASSES[cls_id]}\n"

    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print(f"✓ data.yaml created at {yaml_path}")

    print("\n========================================")
    print(" SUCCESS! Your dataset is ready.")
    print(" Start training with:")
    print(f" yolo train data='{yaml_path.absolute().as_posix()}' model=yolov8n.pt epochs=100 imgsz=640")
    print("========================================")


if __name__ == "__main__":
    prepare_dataset()

 YOLO DATASET PREPARATION PIPELINE

Scanning source directories...
✓ Found 0 images.
✓ Found 0 complete image-label pairs.
❌ Error: No valid data to split. Exiting.


In [ ]:
import torch
from ultralytics import YOLO
import multiprocessing

if __name__ == '__main__':
    # --- 1. HARDWARE CHECK ---
    print("========================================")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Detected:   {torch.cuda.get_device_name(0)}")
        print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("========================================\n")

    multiprocessing.freeze_support()

    # --- 2. LOAD PRE-TRAINED MODEL ---
    # .yaml which starts from random weights
    model = YOLO('yolov8n.yaml')  # Nano — good balance of speed/accuracy for UI detection

    # --- 3. TRAINING ---
    model.train(
        # ── Setup ────────────────────────────────────────────────────────────
        data=r'C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\yolo_training_data\data.yaml',
        epochs=200,          # 15 classes + 100k images needs more than 100 epochs
        patience=30,         # early stop if no improvement for 30 epochs
        imgsz=640,
        device=0,

        # ── Hardware (RTX 3070 8GB VRAM) ─────────────────────────────────────
        batch=32,            # 32 is safe for YOLOv8n at 640p on 8GB
        workers=8,
        cache=True,          # cache images in RAM — big speed boost for 100k images
                             # needs ~12-15GB free RAM; use cache='disk' if RAM is tight

        # ── Learning dynamics ─────────────────────────────────────────────────
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,

        # ── Augmentation ─────────────────────────────────────────────────────
        # Spatial — mostly disabled since your generator already handles this
        fliplr=0.0,          # never mirror UI (bars are left-to-right by design)
        flipud=0.0,          # never flip upside down
        degrees=0.0,         # no rotation — generator handles perspective warp
        perspective=0.0,     # handled in generator
        scale=0.3,           # small additional zoom variation on top of generator
        translate=0.1,       # slight pan — fine for UI

        # Mosaic — KEEP ON: tiles 4 scenes together, increases bar size/position
        # variety the model sees per batch without breaking geometry
        mosaic=1.0,
        close_mosaic=15,     # disable mosaic in the final 15 epochs so the model
                             # fine-tunes on clean single-scene images
        mixup=0.0,           # don't blend two scenes together — too unrealistic for UI

        # Color — allow these to handle monitor/stream variation
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.4,

        # ── Logging ──────────────────────────────────────────────────────────
        project='LolHealthBar',
        name='yolov8n_15cls',
        exist_ok=True,
    )

CUDA Available: True
GPU Detected: NVIDIA GeForce RTX 3070

Ultralytics 8.4.67  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\yolo_training_data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.93

RuntimeError: Dataset 'C://Users/USER/Documents/GitHub/DeeperHealthBarsLeague/LolHealth-Bar/yolo_training_data/data.yaml' error  'C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\yolo_training_data\data.yaml' does not exist